# P01 (basic) — forward kinematics, workspace and the Jacobian matrix

**Module 21 — Robotics 1**

The first question of every robot is: **"Where is my hand?"** The answer is the
**forward kinematics** — a product of homogeneous transformations along the joint chain
(building directly on module 19). Its **derivative**, the **Jacobian matrix**, answers the
follow-up question: *how does the hand move when I turn the joints?* — and reveals where the arm
becomes **singular**.

### Goal
After this project you can …
- set up the general **DH transformation matrix** (ch. 4 of the script),
- evaluate the **kinematic chain** as a matrix product and check it against the closed-form formula,
- make the **workspace** of an arm visible by sampling,
- set up the **Jacobian matrix** analytically, verify it numerically and find and interpret its
  **singularities** ($\det\mathbf J = l_1 l_2 \sin q_2$).

### Format
Jupyter notebook — kinematics thrives on the interplay of formula, number and drawing.

### Prior knowledge
Homogeneous 4x4 transformations (**module 19**), ch. 3-5 of the module 21 script, partial derivatives.

### Tasks
Most of it is given; at the `# TODO` spots you build the core blocks. The solution is in `solution/`.

## Setup
Only `numpy` and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## Part A — the DH transformation matrix

The Denavit-Hartenberg convention describes the transition from frame $i-1$ to frame $i$ with
**four** parameters $(\theta, d, a, \alpha)$ as
$\mathrm{Rot}_z(\theta)\,\mathrm{Trans}_z(d)\,\mathrm{Trans}_x(a)\,\mathrm{Rot}_x(\alpha)$:

$$^{i-1}\mathbf T_i=\begin{pmatrix}
\cos\theta & -\sin\theta\cos\alpha & \sin\theta\sin\alpha & a\cos\theta\\
\sin\theta & \cos\theta\cos\alpha & -\cos\theta\sin\alpha & a\sin\theta\\
0 & \sin\alpha & \cos\alpha & d\\
0&0&0&1\end{pmatrix}$$

**Your task:** build this matrix. (Hint: either the four individual matrices first, then multiply —
or enter the multiplied-out form directly. Both are valid.)

In [ ]:
def dh_matrix(theta, d, a, alpha):
    # 4x4 DH transformation matrix
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    # TODO: enter the matrix (the rows of the formula above)
    T = ...  # TODO
    return T

# self-check: theta=0,d=0,a=1,alpha=0 -> pure translation by 1 along x
print(dh_matrix(0.0, 0.0, 1.0, 0.0))
# self-check: 90 degrees about z, no translation
print(dh_matrix(np.pi/2, 0.0, 0.0, 0.0).round(6))

**Expectation.** The first matrix is the identity with a 1 in the translation column $x$
(a pure translation by 1 along $x$). The second is the 90 degree rotation about $z$
(first column $(0,1,0)$, second $(-1,0,0)$).

## Part B — the kinematic chain (forward kinematics)

For a **planar** arm with link lengths $l_1,\dots,l_n$ and revolute joints we have
$d_i=0$, $\alpha_i=0$, $a_i=l_i$, $\theta_i=q_i$. The pose of the end effector is the
**product of the chain**:

$$^{0}\mathbf T_n(\mathbf q) = {}^{0}\mathbf T_1(q_1)\,{}^{1}\mathbf T_2(q_2)\cdots{}^{n-1}\mathbf T_n(q_n)$$

**Your task:** multiply the chain out and collect the **joint positions** along the way
(for the drawing). Afterwards we check against the closed-form formula
$x=l_1\cos q_1 + l_2\cos(q_1{+}q_2)$, $y=l_1\sin q_1+l_2\sin(q_1{+}q_2)$.

In [ ]:
def fk_joints(q, lengths):
    # returns the (n+1, 2) joint positions: base (0,0) + every joint + the end effector
    T = np.eye(4)
    pts = [np.zeros(2)]
    for qi, li in zip(q, lengths):
        # TODO: append the next DH matrix (planar: d=0, a=li, alpha=0) and
        #       read the x/y position from the translation column of T
        ...  # TODO
        pts.append(T[:2, 3].copy())
    return np.array(pts)

def ee_position(q, lengths):
    # end effector position = the last point of the chain   [given]
    return fk_joints(q, lengths)[-1]

# verification against the closed-form formula (2 links)
L = [1.0, 1.0]
for q in [[0.0, 0.0], [0.7, 1.1], [-0.4, 2.0]]:
    q = np.array(q)
    closed = np.array([L[0]*np.cos(q[0]) + L[1]*np.cos(q[0]+q[1]),
                       L[0]*np.sin(q[0]) + L[1]*np.sin(q[0]+q[1])])
    print(f"q={q}  chain={ee_position(q, L).round(6)}  closed form={closed.round(6)}  "
          f"equal={np.allclose(ee_position(q, L), closed)}")

**Expectation.** For all three configurations `equal=True` must come out — the matrix chain and
the formula derived by hand describe the same thing. At $q=(0,0)$ the arm is stretched along $x$:
the end effector sits at $(2,0)$.

Note the $\cos(q_1{+}q_2)$ in the closed-form formula: **the angles add up along the chain**,
because every link rotates along with all the previous ones.

## Part C — the workspace

The **workspace** is the set of all reachable end effector positions. We make it visible by
**sampling**: many random joint settings, plot the end effector.
(Cell given.)

In [ ]:
Q = rng.uniform(-np.pi, np.pi, size=(4000, 2))
P = np.array([ee_position(q, L) for q in Q])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].scatter(P[:, 0], P[:, 1], s=2, alpha=0.25)
axes[0].set_title(f"workspace 2-joint (l1={L[0]}, l2={L[1]})")
axes[0].set_aspect("equal"); axes[0].set_xlabel("x"); axes[0].set_ylabel("y")

# unequal lengths -> annulus with a hole (inner radius |l1-l2|)
L2 = [1.2, 0.5]
P2 = np.array([ee_position(q, L2) for q in Q])
axes[1].scatter(P2[:, 0], P2[:, 1], s=2, alpha=0.25, color="darkorange")
axes[1].set_title(f"workspace (l1={L2[0]}, l2={L2[1]}) - hole of radius |l1-l2|={abs(L2[0]-L2[1]):.1f}")
axes[1].set_aspect("equal"); axes[1].set_xlabel("x")
plt.tight_layout(); plt.show()

print(f"max reach measured: {np.linalg.norm(P2, axis=1).max():.3f}  (l1+l2 = {sum(L2)})")
print(f"min reach measured: {np.linalg.norm(P2, axis=1).min():.3f}  (|l1-l2| = {abs(L2[0]-L2[1])})")

**Expectation.** With equal lengths ($l_1=l_2$) the workspace is a **full disc** of radius
$l_1+l_2=2$ (the inner radius $|l_1-l_2|=0$ vanishes). With **unequal** lengths an
**annulus with a hole** appears: outer radius $l_1+l_2=1.7$, inner radius $|l_1-l_2|=0.7$ —
points closer than 0.7 to the base are **unreachable**, no matter how you set the joints.

## Part D — the Jacobian matrix and its singularities

The Jacobian matrix is the derivative of the forward kinematics, $\dot{\mathbf x}=\mathbf J(\mathbf q)\dot{\mathbf q}$.
For the planar 2-joint arm, differentiating the formulas of part B gives:

$$\mathbf J = \begin{pmatrix}
-l_1\sin q_1 - l_2\sin(q_1{+}q_2) & -l_2\sin(q_1{+}q_2)\\
\;\;\,l_1\cos q_1 + l_2\cos(q_1{+}q_2) & \;\;\,l_2\cos(q_1{+}q_2)\end{pmatrix}$$

**Your task:** enter $\mathbf J$. Afterwards we compare it with a **numerical** derivative
(central difference quotient) — the best test for any analytic derivative.

In [ ]:
def jacobian_analytic(q, lengths):
    # analytic Jacobian matrix (2x2) for the planar 2-joint arm
    q1, q2 = q
    l1, l2 = lengths
    s1, c1 = np.sin(q1), np.cos(q1)
    s12, c12 = np.sin(q1+q2), np.cos(q1+q2)
    # TODO: enter the 2x2 matrix
    J = ...  # TODO
    return J

def jacobian_numeric(q, lengths, h=1e-6):
    # numerical Jacobian via the central difference quotient   [given]
    q = np.asarray(q, float)
    cols = []
    for k in range(len(q)):
        e = np.zeros(len(q)); e[k] = h
        cols.append((ee_position(q+e, lengths) - ee_position(q-e, lengths)) / (2*h))
    return np.column_stack(cols)

q_test = np.array([0.7, 1.1])
Ja, Jn = jacobian_analytic(q_test, L), jacobian_numeric(q_test, L)
print("analytic :\n", Ja)
print("numerical:\n", Jn)
print("matching:", np.allclose(Ja, Jn, atol=1e-6))
print()
# determinant and singularities
print(f"det J = {np.linalg.det(Ja):.6f}   l1*l2*sin(q2) = {L[0]*L[1]*np.sin(q_test[1]):.6f}")
for q2 in [0.0, np.pi]:
    print(f"  q2={q2:.3f} (stretched/folded): det J = "
          f"{np.linalg.det(jacobian_analytic([0.4, q2], L)):.2e}  -> SINGULAR")

**Expectation / self-check.**
- `matching: True` — the analytic and the numerical derivative are identical (down to ~$10^{-6}$).
- $\det\mathbf J$ agrees exactly with $l_1 l_2\sin q_2$.
- At $q_2=0$ (**stretched**) and $q_2=\pi$ (**folded**) $\det\mathbf J=0$: a **singularity**.

**What that means physically:** in the stretched state the end effector can **no longer move
radially outwards** — one direction of motion is lost. Closer to the singularity you would need
ever larger joint velocities for the same hand motion. That is exactly what makes the naive
pseudoinverse unstable in inverse kinematics (the topic of **P02**).

## Part E — making the manipulability visible

The **manipulability** $w=\sqrt{\det(\mathbf J\mathbf J^\top)}$ (Yoshikawa) measures how "well"
the arm can move in a configuration; it drops to 0 in singularities.
We draw it over the configuration space $(q_1,q_2)$ — plus a few arm poses.
(Cells given.)

In [ ]:
q1s = np.linspace(-np.pi, np.pi, 200)
q2s = np.linspace(-np.pi, np.pi, 200)
W = np.zeros((len(q2s), len(q1s)))
for i, q2 in enumerate(q2s):
    for j, q1 in enumerate(q1s):
        J = jacobian_analytic([q1, q2], L)
        W[i, j] = np.sqrt(max(np.linalg.det(J @ J.T), 0.0))

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.pcolormesh(q1s, q2s, W, shading="auto", cmap="viridis")
ax.set_xlabel("q1 [rad]"); ax.set_ylabel("q2 [rad]")
ax.set_title("manipulability w over the C-space\n(dark bands at q2=0, +-pi = singularities)")
plt.colorbar(im, ax=ax, label="w"); plt.tight_layout(); plt.show()
print("w depends ONLY on q2 (vertical stripes) - consistent with det J = l1*l2*sin(q2).")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for q, col, lab in [([0.0, 0.0], "red", "stretched (singular)"),
                    ([0.5, 1.2], "steelblue", "generic"),
                    ([1.2, np.pi], "darkorange", "folded (singular)")]:
    pts = fk_joints(np.array(q), L)
    ax.plot(pts[:, 0], pts[:, 1], "o-", color=col, lw=2.5, ms=7, label=lab)
circ = plt.Circle((0, 0), sum(L), fill=False, ls="--", color="gray", label="max reach")
ax.add_patch(circ)
ax.set_aspect("equal"); ax.set_xlim(-2.3, 2.3); ax.set_ylim(-2.3, 2.3)
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_title("arm poses")
plt.tight_layout(); plt.show()

## Conclusion

You have built the kinematic foundation:
- the **DH matrix** and the **chain** (verified against the closed-form formula),
- the **workspace** (an annulus — with a hole for unequal link lengths),
- the **Jacobian matrix** (analytic, cross-checked numerically) and its **singularities**
  at $q_2 = 0, \pi$.

Everything is now ready for the inverse: **P02 (medium)** solves the **inverse kinematics** —
analytically (with both elbow solutions) and numerically via the Jacobian. There you will see why
the singularities of part D blow up the naive pseudoinverse and how **damped least squares**
rescues it.